# Analysis Code for Entity Impact Prediction

This notebook contains custom analysis and visualization code developed for the Entity Impact Prediction project.

**Authors**: Our own implementation
**Purpose**: Result analysis and visualization of entity impact prediction outcomes.


去重

In [ ]:
import pandas as pd
import os

def remove_duplicate_entities(file_path):
    # 读取CSV文件
    df = pd.read_csv(file_path, sep=',')
    
    # 清理列名
    df.columns = df.columns.str.strip()
    
    # 创建不区分大小写的实体名称列
    df['实体名称_小写'] = df['实体名称'].str.lower()
    
    # 基于小写实体名称删除重复，保留第一次出现
    df_deduplicated = df.drop_duplicates(subset=['实体名称_小写'], keep='first')
    
    # 删除临时列
    df_deduplicated = df_deduplicated.drop(columns=['实体名称_小写'])
    
    # 保存处理后的文件
    output_path = file_path.replace('.csv', '_deduplicated.csv')
    df_deduplicated.to_csv(output_path, sep=',', index=False, encoding='utf-8')
    
    print(f"处理完成！原始数据: {len(df)}行，删除重复后: {len(df_deduplicated)}行")
    print(f"文件已保存到: {output_path}")

if __name__ == "__main__":
    file_path = r"D:\Impact4Cast-main\entities\extracted_entities.csv"
    remove_duplicate_entities(file_path)

删去括号

In [ ]:
import csv
import re

def clean_entities(input_file, output_file):
    """
    清洗实体名称，删除括号及其内容
    """
    cleaned_entities = []
    
    with open(input_file, 'r', encoding='utf-8') as file:
        reader = csv.reader(file)
        header = next(reader)  # 读取表头
        
        for row in reader:
            if len(row) >= 2:
                entity_name = row[1].strip()
                # 删除括号及其内容（支持英文括号）
                # 匹配所有括号及其内容，包括嵌套括号
                cleaned_name = re.sub(r'\s*\([^)]*\)', '', entity_name)
                cleaned_name = cleaned_name.strip()
                
                if cleaned_name:  # 只保留非空实体
                    cleaned_entities.append([row[0], cleaned_name, row[2], row[3], row[4]])
    
    # 保存清洗后的文件
    with open(output_file, 'w', encoding='utf-8-sig', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['文档ID', '实体名称', '实体类别', '开始位置', '结束位置'])
        writer.writerows(cleaned_entities)
    
    print(f"原始实体数: {len(cleaned_entities)}")
    print(f"清洗后保存到: {output_file}")

if __name__ == "__main__":
    input_file = r"D:\Impact4Cast-main\entities\extracted_entities_deduplicated.csv"
    output_file = r"D:\Impact4Cast-main\entities\extracted_entities_cleaned.csv"
    
    clean_entities(input_file, output_file)

清洗复数形式

In [ ]:
import csv
import re

def is_plural(word):
    """
    判断单词是否为复数形式
    规则：
    1. 以s结尾
    2. 不是以ss, sss结尾（如process, class）
    3. 不是缩写（全大写或包含数字）
    4. 单数形式存在于实体列表中
    """
    if not word.endswith('s'):
        return False
    
    # 排除以ss, sss结尾的单词（如process, class, business）
    if word.endswith('ss') or word.endswith('sss'):
        return False
    
    # 排除全大写的缩写（如LLMs, GNNs需要特殊处理）
    # 但如果只有最后一个字母是小写s，前面都是大写，则认为是复数缩写
    if word[:-1].isupper() and word[-1] == 's':
        return True
    
    # 排除包含数字的
    if any(c.isdigit() for c in word):
        return False
    
    return True

def get_singular(word):
    """
    获取单词的单数形式
    """
    if not is_plural(word):
        return word
    
    # 处理以s结尾的复数
    if word.endswith('ies'):
        return word[:-3] + 'y'
    elif word.endswith('es'):
        return word[:-2]
    elif word.endswith('s'):
        return word[:-1]
    
    return word

def clean_plural_entities(input_file, output_file):
    """
    删除复数形式的实体，保留单数形式
    """
    # 读取所有实体
    entities = []
    with open(input_file, 'r', encoding='utf-8') as file:
        reader = csv.reader(file)
        header = next(reader)
        for row in reader:
            if len(row) >= 2:
                entities.append(row)
    
    # 构建实体名称集合
    entity_names = [row[1] for row in entities]
    entity_set = set(entity_names)
    
    # 找出需要保留的实体（优先保留单数形式）
    keep_indices = []
    remove_names = set()
    
    for i, row in enumerate(entities):
        entity_name = row[1]
        
        # 如果已经被标记删除，跳过
        if entity_name in remove_names:
            continue
        
        # 判断是否为复数形式
        if is_plural(entity_name):
            singular = get_singular(entity_name)
            
            # 如果单数形式存在于实体列表中，则删除复数形式，保留单数
            if singular in entity_set:
                remove_names.add(entity_name)
                continue
            else:
                # 如果单数形式不存在，保留复数形式
                keep_indices.append(i)
        else:
            # 单数形式，保留
            keep_indices.append(i)
    
    # 构建保留的实体列表
    cleaned_entities = [entities[i] for i in keep_indices]
    
    # 保存清洗后的文件
    with open(output_file, 'w', encoding='utf-8-sig', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(cleaned_entities)
    
    print(f"原始实体数: {len(entities)}")
    print(f"删除的复数实体数: {len(remove_names)}")
    print(f"清洗后实体数: {len(cleaned_entities)}")
    print(f"\n删除的复数实体示例:")
    for name in list(remove_names)[:20]:
        print(f"  {name} -> {get_singular(name)}")
    
    return cleaned_entities

def main():
    input_file = r"D:\Impact4Cast-main\entities\extracted_entities_cleaned.csv"
    output_file = r"D:\Impact4Cast-main\entities\extracted_entities_singular.csv"
    
    cleaned_entities = clean_plural_entities(input_file, output_file)
    print(f"\n清洗完成！结果保存到: {output_file}")

if __name__ == "__main__":
    main()

删除不必要的符号

In [ ]:
import csv
import re

def clean_special_characters(input_file, output_file):
    """
    删除实体名称中的特殊符号: ~ " $ % \ /
    保留: & . - + '
    """
    # 定义要删除的字符集
    chars_to_remove = set('~"$%\\/')
    
    # 读取所有实体
    cleaned_entities = []
    
    with open(input_file, 'r', encoding='utf-8') as file:
        reader = csv.reader(file)
        header = next(reader)
        
        for row in reader:
            if len(row) >= 5:
                entity_name = row[1]
                
                # 删除指定字符
                cleaned_name = ''.join(char for char in entity_name if char not in chars_to_remove)
                
                # 清理可能产生的多余空格
                cleaned_name = re.sub(r'\s+', ' ', cleaned_name).strip()
                
                if cleaned_name:  # 只保留非空实体
                    row[1] = cleaned_name
                    cleaned_entities.append(row)
    
    # 保存清洗后的文件
    with open(output_file, 'w', encoding='utf-8-sig', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(cleaned_entities)
    
    print(f"原始实体数: {len(cleaned_entities)}")
    print(f"清洗后保存到: {output_file}")
    
    

def main():
    input_file = r"D:\Impact4Cast-main\entities\extracted_entities_singular.csv"
    output_file = r"D:\Impact4Cast-main\entities\extracted_entities_cleaned_final.csv"
    
    clean_special_characters(input_file, output_file)
    print("\n处理完成！")

if __name__ == "__main__":
    main()

频次统计

In [ ]:
import sys
!{sys.executable} -m pip install pyahocorasick

In [ ]:
# 首先，在Jupyter中安装pyahocorasick到当前环境
import sys
!{sys.executable} -m pip install pyahocorasick

# 然后导入
from ahocorasick import Automaton

In [ ]:
!pip install pyahocorasick

频次统计和文档频次百分比降序排列

In [ ]:
import csv
import json
from collections import defaultdict
from tqdm import tqdm
import sys
import os
import re
from ahocorasick import Automaton

def read_entities(file_path):
    """读取去重后的实体CSV文件，保留原始实体名称（包括大小写和特殊字符）"""
    print(f"Reading entities from: {file_path}")
    
    entities = []
    # 尝试常见的中文编码
    encodings = ['utf-8', 'gbk', 'gb2312', 'gb18030', 'latin-1', 'cp1252']
    
    for enc in encodings:
        try:
            with open(file_path, 'r', encoding=enc) as file:
                reader = csv.reader(file)
                header = next(reader)
                
                for row in reader:
                    if len(row) >= 2:
                        entity_name = row[1].strip()
                        if entity_name:
                            entities.append(entity_name)
            
            print(f"Successfully read with encoding: {enc}")
            print(f"Loaded {len(entities)} unique entities.")
            print(f"Sample entities: {entities[:5]}")
            return entities
            
        except (UnicodeDecodeError, UnicodeError):
            continue
    
    raise Exception("Failed to read entities file with any encoding")

def read_papers(file_path):
    """读取论文JSON数据"""
    print(f"Reading papers from: {file_path}")
    
    encodings = ['utf-8', 'gbk', 'gb2312', 'gb18030']
    
    for enc in encodings:
        try:
            with open(file_path, 'r', encoding=enc) as file:
                papers = json.load(file)
            print(f"Successfully read with encoding: {enc}")
            print(f"Loaded {len(papers)} papers.")
            return papers
        except (UnicodeDecodeError, json.JSONDecodeError):
            continue
    
    raise Exception("Failed to read papers file with any encoding")

def count_entity_document_frequency_ac(entities, papers):
    """
    使用AC自动机统计实体在论文中的文档频次（DF）
    严格区分大小写，保留实体名称中的特殊字符
    """
    print("Building AC automaton...")
    
    # 创建AC自动机
    automaton = Automaton()
    entity_lengths = {}
    
    # 添加所有实体到自动机（保持原始大小写和特殊字符）
    for entity in entities:
        automaton.add_word(entity, entity)  # 直接使用原始实体名称，不转换大小写
        entity_lengths[entity] = len(entity)
    
    # 构建自动机
    automaton.make_automaton()
    print(f"AC automaton built with {len(entities)} patterns.")
    
    # 定义单词边界字符（扩展以包含更多特殊字符）
    word_boundaries = set(" ,.;:!?()[]{}\"'-\n\t\r")
    
    print("Counting entity document frequency...")
    
    # 存储每个实体出现的文档集合
    entity_docs = defaultdict(set)
    total_papers = len(papers)
    
    # 使用tqdm显示进度条
    for idx, paper in tqdm(enumerate(papers, 1), total=total_papers, desc="Processing papers", ncols=100):
        # 合并标题和摘要，保持原始大小写（不转换为小写）
        title = paper.get("title", "")
        abstract = paper.get("abstract", "")
        text = title + " " + abstract  # 保持原始大小写
        
        # 记录当前文档中匹配到的实体
        matched_entities = set()
        
        # 使用AC自动机查找所有可能匹配（区分大小写）
        for end_index, entity in automaton.iter(text):
            entity_len = entity_lengths[entity]
            start_index = end_index - entity_len + 1
            
            # 边界验证：检查匹配词的边界
            # 检查左边边界
            if start_index > 0 and text[start_index - 1] not in word_boundaries:
                continue
            # 检查右边边界
            if end_index + 1 < len(text) and text[end_index + 1] not in word_boundaries:
                continue
            
            # 记录匹配到的实体
            matched_entities.add(entity)
        
        # 为每个匹配到的实体记录当前文档ID
        for entity in matched_entities:
            entity_docs[entity].add(idx)
    
    print(f"Counting complete. Processed {total_papers} papers.")
    
    # 计算每个实体的文档频次
    entity_df = {}
    for entity, doc_set in entity_docs.items():
        entity_df[entity] = len(doc_set)
    
    total_matches = sum(entity_df.values())
    entities_with_matches = sum(1 for freq in entity_df.values() if freq > 0)
    print(f"Total document-entity pairs: {total_matches}")
    print(f"Entities with matches: {entities_with_matches}/{len(entities)}")
    
    return entity_df

def save_results_with_percentage(results, output_file_path, total_docs):
    """
    保存统计结果，包含文档频次、百分比和累加百分比
    删除频次小于1的实体，按频次降序排序
    """
    # 删除频次小于1的实体
    filtered_results = {entity: freq for entity, freq in results.items() if freq >= 1}
    
    # 按频次降序排序
    sorted_results = sorted(filtered_results.items(), key=lambda x: x[1], reverse=True)
    
    # 计算总频数之和
    total_freq_sum = sum(filtered_results.values())
    
    print(f"\nStatistics after filtering:")
    print(f"Entities with frequency >= 1: {len(filtered_results)}")
    print(f"Total frequency sum: {total_freq_sum}")
    
    # 保存为CSV文件
    output_csv = output_file_path.replace('.txt', '.csv')
    
    # 确保输出目录存在
    output_dir = os.path.dirname(output_csv)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    try:
        with open(output_csv, 'w', encoding='utf-8-sig', newline='') as file:
            writer = csv.writer(file)
            # 写入表头
            writer.writerow(['Entity Name', 'Document Frequency', 'Percentage (%)', 'Cumulative Percentage (%)'])
            
            # 计算累加百分比
            cumulative_percentage = 0.0
            
            # 写入数据
            for entity, freq in sorted_results:
                try:
                    # 计算百分比
                    percentage = (freq / total_freq_sum) * 100
                    # 累加百分比
                    cumulative_percentage += percentage
                    # 写入行
                    writer.writerow([entity, freq, f"{percentage:.4f}%", f"{cumulative_percentage:.4f}%"])
                except Exception as e:
                    print(f"Error writing entity '{entity}': {e}")
                    continue
        
        print(f"\nResults saved to: {output_csv}")
        
    except Exception as e:
        print(f"Error saving CSV file: {e}")
    
    # 同时保存txt格式
    try:
        with open(output_file_path, 'w', encoding='utf-8') as file:
            for entity, freq in sorted_results:
                file.write(f"{entity}\t{freq}\n")
        print(f"TXT results saved to: {output_file_path}")
    except Exception as e:
        print(f"Error saving TXT file: {e}")
    
    return filtered_results, total_freq_sum

def main():
    """主程序"""
    print("=" * 60)
    print("Entity Document Frequency Counter (AC Automaton Version)")
    print("=" * 60)
    
    # 设定文件路径
    entity_file = r"D:\Impact4Cast-main\entities\extracted_entities_cleaned.csv"
    paper_file = r"D:\Impact4Cast-main\arxiv_cs_original.json"
    output_file = r"D:\Impact4Cast-main\entities\entity_document_frequency.txt"
    
    # 确保输出目录存在
    output_dir = os.path.dirname(output_file)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # 如果去重后的文件不存在，尝试使用原始文件
    if not os.path.exists(entity_file):
        print(f"去重文件不存在，尝试使用原始文件...")
        entity_file = r"D:\Impact4Cast-main\entities\extracted_entities.csv"
    
    # 检查文件是否存在
    if not os.path.exists(entity_file):
        print(f"错误: 实体文件不存在: {entity_file}")
        return
    
    if not os.path.exists(paper_file):
        print(f"错误: 论文文件不存在: {paper_file}")
        return
    
    # 读取数据
    entities = read_entities(entity_file)
    papers = read_papers(paper_file)
    
    print(f"\nStarting document frequency counting...")
    print(f"Unique entities: {len(entities)}, Papers: {len(papers)}")
    
    # 统计实体文档频次
    entity_df = count_entity_document_frequency_ac(entities, papers)
    
    # 保存结果
    filtered_results, total_freq_sum = save_results_with_percentage(
        entity_df, output_file, len(papers)
    )
    
    # 打印前10个结果作为示例
    if filtered_results:
        print("\nTop 10 entities by document frequency:")
        sorted_items = sorted(filtered_results.items(), key=lambda x: x[1], reverse=True)[:10]
        cumulative = 0.0
        for entity, freq in sorted_items:
            percentage = (freq / total_freq_sum) * 100
            cumulative += percentage
            print(f"  {entity}: {freq} ({percentage:.4f}%, Cumulative: {cumulative:.4f}%)")
    
    print("\n处理完成！")

if __name__ == "__main__":
    main()

添加对应类别标签

In [ ]:
import pandas as pd

# 读取文件
entities_df = pd.read_csv(r"D:\Impact4Cast-main\entities\extracted_entities_cleaned_final.csv", encoding='utf-8-sig')
predictions_df = pd.read_csv(r"D:\Impact4Cast-main\entities\results\all_predictions_with_entities.csv", encoding='utf-8-sig')

# 创建实体ID到类别的映射字典
entity_to_category = dict(zip(entities_df['实体名称'], entities_df['实体类别']))

# 为concept1和concept2添加类别列
predictions_df['concept1_category'] = predictions_df['concept1_keyword'].map(entity_to_category)
predictions_df['concept2_category'] = predictions_df['concept2_keyword'].map(entity_to_category)

# 保存结果
output_path = r"D:\Impact4Cast-main\entities\results\all_predictions_with_categories.csv"
predictions_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"处理完成！结果已保存到: {output_path}")
print(f"共处理 {len(predictions_df)} 条实体对记录")

概念对得分统计分析

In [ ]:
!pip install sys

In [ ]:
import sys
!{sys.executable} -m pip install matplotlib seaborn

In [ ]:
!pip install matplotlib seaborn

In [ ]:
import pandas as pd

df = pd.read_csv(r"D:\Impact4Cast-main\entities\extracted_entities_cleaned_final.csv", encoding='utf-8-sig')

result = df['实体类别'].value_counts().to_frame('Count')
result['Percentage (%)'] = df['实体类别'].value_counts(normalize=True).mul(100).round(2)
print(result)
print(f"\nTotal: {len(df)}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")
sns.set_palette("husl")

# Read data
file_path = r"D:\Impact4Cast-main\entities\results\all_predictions_with_categories.csv"
df = pd.read_csv(file_path, encoding='utf-8-sig')

# Get categories
categories = sorted(set(df['concept1_category'].unique()) | set(df['concept2_category'].unique()))

# Create symmetric category combination matrix
def create_symmetric_category_matrix(df):
    categories = sorted(set(df['concept1_category'].unique()) | set(df['concept2_category'].unique()))
    
    matrix_data = []
    for i, cat1 in enumerate(categories):
        row = []
        for j, cat2 in enumerate(categories):
            if i <= j:
                subset1 = df[(df['concept1_category'] == cat1) & (df['concept2_category'] == cat2)]
                subset2 = df[(df['concept1_category'] == cat2) & (df['concept2_category'] == cat1)]
                subset = pd.concat([subset1, subset2])
                
                if len(subset) > 0:
                    avg_score = subset['score'].mean()
                    count = len(subset)
                    std_score = subset['score'].std() if len(subset) > 1 else 0
                else:
                    avg_score = np.nan
                    count = 0
                    std_score = 0
                
                row.append({'avg_score': avg_score, 'count': count, 'std': std_score})
            else:
                row.append(matrix_data[j][i])
        matrix_data.append(row)
    
    return categories, matrix_data

categories, matrix_data = create_symmetric_category_matrix(df)

avg_score_matrix = pd.DataFrame(
    [[cell['avg_score'] for cell in row] for row in matrix_data],
    index=categories, columns=categories
)

# ==================== Average Score Heatmap (Separate) ====================
# Create figure for average score heatmap only
fig, ax = plt.subplots(figsize=(8, 6))

# Calculate color range based on data distribution
vmin = avg_score_matrix.values.min()
vmax = avg_score_matrix.values.max()
# Add small padding to make colors more sensitive
vmin_padded = vmin - (vmax - vmin) * 0.05
vmax_padded = vmax + (vmax - vmin) * 0.05

# Use a more sensitive colormap with more color gradations
im = ax.imshow(avg_score_matrix.values, cmap='RdYlGn_r', aspect='auto', 
               vmin=vmin_padded, vmax=vmax_padded, interpolation='bilinear')

# Set ticks and labels
ax.set_xticks(range(len(categories)))
ax.set_yticks(range(len(categories)))
ax.set_xticklabels(categories, rotation=45, ha='right', fontsize=12)
ax.set_yticklabels(categories, fontsize=12)
ax.set_title('Average Score Matrix (Symmetric)', fontsize=16, fontweight='bold')
ax.set_xlabel('Category', fontsize=13)
ax.set_ylabel('Category', fontsize=13)

# Add text annotations with more precision
for i in range(len(categories)):
    for j in range(len(categories)):
        if not np.isnan(avg_score_matrix.values[i, j]):
            # Determine text color based on background
            value = avg_score_matrix.values[i, j]
            # Normalize value for color brightness calculation
            norm_value = (value - vmin) / (vmax - vmin) if vmax > vmin else 0.5
            # Use white text for dark backgrounds, black for light backgrounds
            if norm_value > 0.6:
                text_color = 'white'
            else:
                text_color = 'black'
            ax.text(j, i, f'{value:.6f}', ha="center", va="center", 
                   color=text_color, fontsize=10, fontweight='bold')

# Add colorbar with more ticks for better sensitivity
cbar = plt.colorbar(im, ax=ax, label='Average Score', fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=10)

# Save the figure
output_path = r"D:\Impact4Cast-main\entities\results\average_score_heatmap.png"
plt.tight_layout()
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f"Average score heatmap saved to: {output_path}")
print(f"Score range: {vmin:.6f} to {vmax:.6f}")
print(f"Color scale range: {vmin_padded:.6f} to {vmax_padded:.6f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")
sns.set_palette("husl")

# Directly use the provided statistical data with actual average scores
data = [
    {'Combination': 'method — task', 'Count': 268207, 'Percentage': 26.82, 'Average_Score': 0.451683550944976},
    {'Combination': 'method — method', 'Count': 238326, 'Percentage': 23.83, 'Average_Score': 0.451701455249992},
    {'Combination': 'dataset — method', 'Count': 209839, 'Percentage': 20.98, 'Average_Score': 0.451670681665218},
    {'Combination': 'dataset — task', 'Count': 117049, 'Percentage': 11.70, 'Average_Score': 0.451734064263642},
    {'Combination': 'task — task', 'Count': 74041, 'Percentage': 7.40, 'Average_Score': 0.451716336796281},
    {'Combination': 'dataset — dataset', 'Count': 46223, 'Percentage': 4.62, 'Average_Score': 0.451769339916589},
    {'Combination': 'method — metric', 'Count': 23012, 'Percentage': 2.30, 'Average_Score': 0.451830314605392},
    {'Combination': 'metric — task', 'Count': 12764, 'Percentage': 1.28, 'Average_Score': 0.451561543113404},
    {'Combination': 'dataset — metric', 'Count': 9990, 'Percentage': 1.00, 'Average_Score': 0.451596978053247},
    {'Combination': 'metric — metric', 'Count': 549, 'Percentage': 0.05, 'Average_Score': 0.451556638018898},
]

# Convert to DataFrame
comb_df = pd.DataFrame(data)

# Sort by Count descending
comb_df = comb_df.sort_values('Count', ascending=False).reset_index(drop=True)

# ==================== Score vs Frequency Scatter Plot (Square, Small) ====================
fig, ax = plt.subplots(figsize=(8, 6))

# Create scatter plot
scatter = ax.scatter(comb_df['Count'], comb_df['Average_Score'], 
                     c=comb_df['Average_Score'], cmap='RdYlGn_r', 
                     s=150, alpha=0.7, edgecolors='black', linewidth=1.5)

ax.set_xlabel('Number of Pairs', fontsize=10, fontweight='bold')
ax.set_ylabel('Average Score', fontsize=10, fontweight='bold')
# 标题已删除
ax.grid(True, alpha=0.3, linestyle='--')

# Add labels for each point with percentage
for _, row in comb_df.iterrows():
    label = f"{row['Combination']}\n({row['Percentage']:.2f}%)"
    ax.annotate(label, (row['Count'], row['Average_Score']),
                xytext=(4, 4), textcoords='offset points', 
                fontsize=7, alpha=0.8, fontweight='semibold',
                ha='left', va='bottom')

# Add statistical information box
correlation = comb_df['Count'].corr(comb_df['Average_Score'])
stats_text = f'Correlation: {correlation:.4f}'
ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, 
        fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()

# Save the figure
output_path = r"D:\Impact4Cast-main\entities\results\score_vs_frequency_scatter.png"
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()